In [3]:
import os
import time
import pandas as pd
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt

In [5]:
input_folder = '/kaggle/input/datasets/alwaysashif/motor-sound-detect'
output_folder = '/kaggle/working/Cars_Sounds'
os.makedirs(output_folder, exist_ok=True)

In [6]:
start = time.time()

data = []
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.endswith(".wav"):

            filepath = os.path.join(root, file)
            label = os.path.basename(root)

            try:
                y, sr = librosa.load(filepath, sr=None)
                features = {}
                
                # Time Domain Feature
                features["zcr"] = np.mean(
                    librosa.feature.zero_crossing_rate(y)
                )

                features["rms"] = np.mean(librosa.feature.rms(y=y))

                # Frequency Domain Features
                
                features["spectral_centroid"] = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))

                features["spectral_bandwidth"] = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))

                features["spectral_rolloff"] = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))

                features["spectral_contrast"] = np.mean(librosa.feature.spectral_contrast(y=y, sr=sr),axis=1)

                # MFCC
                
                mfcc = librosa.feature.mfcc(y=y,sr=sr,n_mfcc=13)

                mfcc_mean = np.mean(mfcc, axis=1)


                # Store Features

                row = {}

                row["filename"] = file

                row["label"] = label

                row["zcr"] = features["zcr"]

                row["rms"] = features["rms"]

                row["spectral_centroid"] = features["spectral_centroid"]

                row["spectral_bandwidth"] = features["spectral_bandwidth"]

                row["spectral_rolloff"] = features["spectral_rolloff"]

                # Spectral Contrast
                for i, value in enumerate(features["spectral_contrast"]):
                    row[f"spectral_contrast_{i+1}"] = value

                # MFCC
                for i, value in enumerate(mfcc_mean):
                    row[f"mfcc_{i+1}"] = value
                    
                data.append(row)

            except Exception as e:
                print(f"Error processing {filepath}: {e}")

df = pd.DataFrame(data)
df.to_csv("Petrol_Diesel_Motor_Sound.csv", index=False)
print("Shape:", df.shape)

end = time.time()
print(f"Total time : {end - start: .2f} seconds")

Shape: (320, 27)
Total time :  54.89 seconds


In [7]:
start = time.time()

output_folder = "/kaggle/working/MelSpectrograms"
os.makedirs(output_folder, exist_ok=True)

# Travel to all subfolders
for root, dirs, files in os.walk(input_folder):

    relative_path = os.path.relpath(root, input_folder)
    save_folder = os.path.join(output_folder, relative_path)
    os.makedirs(save_folder, exist_ok=True)

    for file in files:

        if file.lower().endswith(".wav"):
            file_path = os.path.join(root, file)

            # Loading audio
            y, sr = librosa.load(file_path, sr=None)

            # Mel Spectrogram
            mel = librosa.feature.melspectrogram(y=y,sr=sr,n_mels=128)

            mel_db = librosa.power_to_db(mel, ref=np.max)

            # Plotting
            plt.figure(figsize=(3,3))
            librosa.display.specshow(
                mel_db,
                sr=sr,
                cmap="viridis"
            )
            plt.axis("off")

            # image saving
            image_name = file.replace(".wav", ".png")

            plt.savefig(
                os.path.join(save_folder, image_name),
                bbox_inches="tight",
                pad_inches=0
            )

            plt.close()

print("All Mel Spectrogram images have been saved!")

end = time.time()
print(f"Total time : {end - start: .2f} seconds")

All Mel Spectrogram images have been saved!
Total time :  50.15 seconds


In [8]:
import shutil

shutil.make_archive(
    "/kaggle/working/MelSpectrograms",
    "zip",
    "/kaggle/working/MelSpectrograms"
)

print("ZIP file created!")

ZIP file created!
